In [ ]:
import pyvisa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def connect():
    rm = pyvisa.ResourceManager() #verbind
    print("Gevonden VISA-resources:", rm.list_resources())

    resources = [r for r in rm.list_resources() if "USB" in r] 
    if not resources:   #foutmelding
        raise RuntimeError(
        )

    scope = rm.open_resource(resources[0])
    scope.timeout = 10000  # ms
    print("Verbonden met:", scope.query("*IDN?").strip())
    return scope


    """
    mode:
      'NORM' -> schermdata, max ~1200 punten
      'RAW'  -> volledige memory depth 
    """

def read_channel(scope, channel="CHAN1", mode="RAW"):
    scope.write(f":WAV:SOUR {channel}")
    scope.write(f":WAV:MODE {mode}")
    scope.write(":WAV:FORM BYTE")

    # Scaling-parameters opvragen
    xinc = float(scope.query(":WAV:XINC?"))
    xorig = float(scope.query(":WAV:XOR?"))
    yinc = float(scope.query(":WAV:YINC?"))
    yorig = float(scope.query(":WAV:YOR?"))
    yref = float(scope.query(":WAV:YREF?"))

    raw = scope.query_binary_values(":WAV:DATA?", datatype="B", container=np.array)

    voltage = (raw.astype(float) - yorig - yref) * yinc
    time = np.arange(len(voltage)) * xinc + xorig

    return time, voltage


def save_csv(time, voltage, channel, filename=None):
    if filename is None:
        filename = f"scope_{channel}.csv"
    df = pd.DataFrame({"Time": time, "Voltage": voltage})
    df.to_csv(filename, index=False)
    print(f"Opgeslagen als {filename}")
    return df


if __name__ == "__main__":
    scope = connect()

    # --- Kanaal 1 uitlezen ---
    t1, v1 = read_channel(scope, "CHAN1", mode="RAW")
    df1 = save_csv(t1, v1, "CH1")

    scope.close()

    # --- Snel plotje ter controle ---
    plt.figure(figsize=(10, 4))
    plt.plot(df1["Time"], df1["Voltage"], label="CH1")
    plt.xlabel("Tijd (s)")
    plt.ylabel("Spanning (V)")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
import pyvisa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def connect():
    """Zoekt en verbindt automatisch met het eerste USB-instrument."""
    rm = pyvisa.ResourceManager()
    print("Gevonden VISA-resources:", rm.list_resources())

    resources = [r for r in rm.list_resources() if "USB" in r]
    if not resources:
        raise RuntimeError(
            "Geen USB-instrument gevonden. Check de kabel, of installeer NI-VISA."
        )

    scope = rm.open_resource(resources[0])
    scope.timeout = 10000  # ms
    print("Verbonden met:", scope.query("*IDN?").strip())
    return scope


def read_channel(scope, channel="CHAN1", mode="RAW"):
    """
    mode:
      'NORM' -> schermdata, max ~1200 punten (werkt altijd, ook tijdens RUN)
      'RAW'  -> volledige memory depth (zet de scoop eerst op STOP!)
    """
    scope.write(f":WAV:SOUR {channel}")
    scope.write(f":WAV:MODE {mode}")
    scope.write(":WAV:FORM BYTE")

    # Scaling-parameters opvragen (nodig om ruwe bytes om te rekenen)
    xinc = float(scope.query(":WAV:XINC?"))
    xorig = float(scope.query(":WAV:XOR?"))
    yinc = float(scope.query(":WAV:YINC?"))
    yorig = float(scope.query(":WAV:YOR?"))
    yref = float(scope.query(":WAV:YREF?"))

    raw = scope.query_binary_values(":WAV:DATA?", datatype="B", container=np.array)

    voltage = (raw.astype(float) - yorig - yref) * yinc
    time = np.arange(len(voltage)) * xinc + xorig

    # Extra scoop-instellingen, puur informatief (niet nodig voor de berekening)
    metadata = {
        "channel": channel,
        "mode": mode,
        "xincrement_s": xinc,
        "xorigin_s": xorig,
        "yincrement_V": yinc,
        "yorigin": yorig,
        "yreference": yref,
        "npoints": len(voltage),
        "timebase_scale_s_per_div": scope.query(":TIM:SCAL?").strip(),
        "channel_scale_V_per_div": scope.query(f":{channel}:SCAL?").strip(),
        "channel_offset_V": scope.query(f":{channel}:OFFS?").strip(),
        "sample_rate_Sa_s": scope.query(":ACQ:SRAT?").strip(),
    }

    return time, voltage, metadata


def save_csv(time, voltage, metadata, channel, filename=None):
    if filename is None:
        filename = f"scope_{channel}.csv"

    with open(filename, "w") as f:
        # Parameters als commentaarregels bovenaan (pandas negeert deze met comment="#")
        for key, value in metadata.items():
            f.write(f"# {key} = {value}\n")

    df = pd.DataFrame({"Time": time, "Voltage": voltage})
    df.to_csv(filename, index=False, mode="a")
    print(f"Opgeslagen als {filename} (inclusief parameters als # commentaarregels)")
    return df


if __name__ == "__main__":
    scope = connect()

    # RAW-mode geeft de volle memory depth (geen compressie naar ~1200 punten),
    # maar vereist dat de scoop gestopt is. We zetten hem hier zelf op Stop.
    scope.write(":STOP")

    # Groter timeout, want RAW-data kan veel groter zijn dan NORM
    scope.timeout = 30000  # ms

    # --- Kanaal 1 uitlezen (volledige resolutie) ---
    t1, v1, meta1 = read_channel(scope, "CHAN1", mode="RAW")
    print(f"Aantal punten CH1: {len(v1)}")
    df1 = save_csv(t1, v1, meta1, "CH1")

    # --- Kanaal 2 uitlezen (indien gebruikt) ---
    # t2, v2, meta2 = read_channel(scope, "CHAN2", mode="RAW")
    # df2 = save_csv(t2, v2, meta2, "CH2")

    # Optioneel: scoop weer laten lopen na afloop
    # scope.write(":RUN")

    scope.close()

    # --- Snel plotje ter controle ---
    plt.figure(figsize=(10, 4))
    plt.plot(df1["Time"], df1["Voltage"], label="CH1")
    plt.xlabel("Tijd (s)")
    plt.ylabel("Spanning (V)")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
import pyvisa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.signal import find_peaks


def connect():
    rm = pyvisa.ResourceManager()
    print("Gevonden VISA-resources:", rm.list_resources())

    resources = [r for r in rm.list_resources() if "USB" in r] 
    if not resources:
        raise RuntimeError("Geen USB-apparaten gevonden")

    scope = rm.open_resource(resources[0])
    scope.timeout = 100000  # ms
    print("Verbonden met:", scope.query("*IDN?").strip())
    return scope


def get_scope_parameters(scope, channel="CHAN1"):
    """
    Haal alle relevante oscilloscoop-parameters op
    """
    params = {
        'timestamp': datetime.now().isoformat(),
        'channel': channel,
        'xinc': float(scope.query(":WAV:XINC?")),
        'xorig': float(scope.query(":WAV:XOR?")),
        'yinc': float(scope.query(":WAV:YINC?")),
        'yorig': float(scope.query(":WAV:YOR?")),
        'yref': float(scope.query(":WAV:YREF?")),
        'timebase': scope.query(":TIM:SCAL?").strip(),
        'vertical_scale': scope.query(f":{channel}:SCAL?").strip(),
        'vertical_offset': scope.query(f":{channel}:OFFS?").strip(),
        'coupling': scope.query(f":{channel}:COUP?").strip(),
        'probe_attenuation': scope.query(f":{channel}:PROB?").strip(),
    }
    return params


def read_channel_raw(scope, channel="CHAN1"):
    """
    Lees volledige RAW memory depth zonder compressie
    """
    scope.write(f":WAV:SOUR {channel}")
    scope.write(":WAV:MODE RAW")  # Volledige memory
    scope.write(":WAV:FORM BYTE")

    # Scaling-parameters opvragen
    xinc = float(scope.query(":WAV:XINC?"))
    xorig = float(scope.query(":WAV:XOR?"))
    yinc = float(scope.query(":WAV:YINC?"))
    yorig = float(scope.query(":WAV:YOR?"))
    yref = float(scope.query(":WAV:YREF?"))

    raw = scope.query_binary_values(":WAV:DATA?", datatype="B", container=np.array)

    voltage = (raw.astype(float) - yorig - yref) * yinc
    time_array = np.arange(len(voltage)) * xinc + xorig

    print(f"Aantal datapunten: {len(voltage)}")
    print(f"Totale meetduur: {time_array[-1] - time_array[0]:.6e} s")
    print(f"Tijdstap (XINC): {xinc:.6e} s")

    return time_array, voltage


def find_peaks_in_data(voltage, prominence=None):
    """
    Vind pieken in de data
    """
    if prominence is None:
        # Schat prominence op basis van signaal
        prominence = (np.max(voltage) - np.min(voltage)) * 0.1
    
    peaks, properties = find_peaks(voltage, prominence=prominence)
    
    return peaks, properties


def save_measurement_raw(time, voltage, params, peaks, measurement_number, base_folder="measurements"):
    """
    Sla RAW meting en parameters op in CSV-bestanden
    """
    import os
    
    if not os.path.exists(base_folder):
        os.makedirs(base_folder)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    data_filename = f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_RAW_data.csv"
    params_filename = f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_params.csv"
    peaks_filename = f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_peaks.csv"
    
    # Sla volledige data op (GEEN compressie)
    df_data = pd.DataFrame({"Time": time, "Voltage": voltage})
    df_data.to_csv(data_filename, index=False)
    print(f"\nData opgeslagen: {data_filename}")
    print(f"Aantal punten in CSV: {len(df_data)}")
    
    # Sla parameters op
    df_params = pd.DataFrame([params])
    df_params.to_csv(params_filename, index=False)
    print(f"Parameters opgeslagen: {params_filename}")
    
    # Sla pieken op
    peaks_data = pd.DataFrame({
        'Peak_Index': peaks,
        'Peak_Time': time[peaks],
        'Peak_Voltage': voltage[peaks]
    })
    peaks_data.to_csv(peaks_filename, index=False)
    print(f"Pieken opgeslagen: {peaks_filename}")
    print(f"Aantal pieken gevonden: {len(peaks)}")
    
    return data_filename, params_filename, peaks_filename


def plot_measurement_raw(time, voltage, peaks, params, measurement_number):
    """
    Plot RAW meting met pieken gemarkeerd
    """
    # Bepaal beste eenheid voor tijdas
    time_max = time[-1]
    if time_max < 1e-3:
        time_display = time * 1e6
        time_unit = "μs"
    elif time_max < 1:
        time_display = time * 1e3
        time_unit = "ms"
    else:
        time_display = time
        time_unit = "s"
    
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(time_display, voltage, label="CH1 (RAW)", linewidth=0.5, color='blue')
    ax.plot(time_display[peaks], voltage[peaks], "rx", markersize=8, label=f"Pieken ({len(peaks)})", linewidth=2)
    
    ax.set_xlabel(f"Tijd ({time_unit})")
    ax.set_ylabel("Spanning (V)")
    ax.set_title(f"Meting {measurement_number} - RAW Data\n"
                 f"Timebase: {params['timebase']}, Verticale schaal: {params['vertical_scale']}\n"
                 f"Datapunten: {len(voltage)} | Pieken: {len(peaks)}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    
    plt.show()
    return fig


if __name__ == "__main__":
    scope = connect()

    print("\n=== RAW DATA METING (GEEN COMPRESSIE) ===\n")
    
    # Lees volledige RAW data
    t, v = read_channel_raw(scope, "CHAN1")
    
    # Haal parameters op
    params = get_scope_parameters(scope, "CHAN1")
    
    # Vind pieken
    peaks, properties = find_peaks_in_data(v)
    
    # Sla alles op
    data_file, params_file, peaks_file = save_measurement_raw(t, v, params, peaks, 1, "measurements")
    
    scope.close()
    
    # Plot met pieken
    plot_measurement_raw(t, v, peaks, params, 1)
    
    print(f"\n✓ Meting voltooid!")
    print(f"Pieken opgeslagen in: {peaks_file}")

In [ ]:
import pyvisa

rm = pyvisa.ResourceManager()
resources = [r for r in rm.list_resources() if "USB" in r]
scope = rm.open_resource(resources[0])

# Haal timebase op
timebase = float(scope.query(":TIM:SCAL?"))
xinc = float(scope.query(":WAV:XINC?"))

print(f"Timebase: {timebase} s/div")
print(f"Tijdstap (XINC): {xinc} s")
print(f"Tijdstap in μs: {xinc * 1e6} μs")

scope.close()

In [ ]:
import pyvisa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.signal import find_peaks, savgol_filter


def connect():
    rm = pyvisa.ResourceManager()
    print("Gevonden VISA-resources:", rm.list_resources())

    resources = [r for r in rm.list_resources() if "USB" in r]
    if not resources:
        raise RuntimeError("Geen USB-apparaten gevonden")

    scope = rm.open_resource(resources[0])
    scope.timeout = 30000  # ms per blok
    print("Verbonden met:", scope.query("*IDN?").strip())
    return scope


def get_scope_parameters(scope, channel="CHAN1"):
    params = {
        'timestamp':         datetime.now().isoformat(),
        'channel':           channel,
        'xinc':              float(scope.query(":WAV:XINC?")),
        'xorig':             float(scope.query(":WAV:XOR?")),
        'yinc':              float(scope.query(":WAV:YINC?")),
        'yorig':             float(scope.query(":WAV:YOR?")),
        'yref':              float(scope.query(":WAV:YREF?")),
        'timebase':          float(scope.query(":TIM:SCAL?")),
        'trigger_offset':    float(scope.query(":TIM:OFFS?")),
        'vertical_scale':    scope.query(f":{channel}:SCAL?").strip(),
        'vertical_offset':   scope.query(f":{channel}:OFFS?").strip(),
        'coupling':          scope.query(f":{channel}:COUP?").strip(),
        'probe_attenuation': scope.query(f":{channel}:PROB?").strip(),
        'memory_depth':      scope.query(":ACQ:MDEP?").strip(),
    }
    return params


def read_channel_raw(scope, channel="CHAN1", chunk_size=10000):
    """
    Lees volledige RAW memory depth van Rigol DS2102A in blokken.

    De Rigol DS2102A gebruikt :ACQ:MDEP voor de memory depth.
    chunk_size : aantal punten per blok (verlaag naar 5000 bij timeout)
    """
    scope.write(f":WAV:SOUR {channel}")
    scope.write(":WAV:MODE RAW")
    scope.write(":WAV:FORM BYTE")

    # Scaling-parameters
    xinc  = float(scope.query(":WAV:XINC?"))
    xorig = float(scope.query(":WAV:XOR?"))
    yinc  = float(scope.query(":WAV:YINC?"))
    yorig = float(scope.query(":WAV:YOR?"))
    yref  = float(scope.query(":WAV:YREF?"))

    # Rigol DS2102A: memory depth via :ACQ:MDEP
    mdep_raw = scope.query(":ACQ:MDEP?").strip()

    # "AUTO" betekent dat de scope zelf kiest — vraag dan het werkelijke aantal op
    if mdep_raw.upper() == "AUTO":
        total_points = int(float(scope.query(":WAV:POIN?")))
        print(f"Memory depth: AUTO → werkelijk {total_points} punten")
    else:
        total_points = int(float(mdep_raw))
        print(f"Memory depth: {total_points} punten")

    # Haal data op in blokken
    raw_all = []
    start   = 1

    while start <= total_points:
        end = min(start + chunk_size - 1, total_points)
        scope.write(f":WAV:STAR {start}")
        scope.write(f":WAV:STOP {end}")

        chunk = scope.query_binary_values(":WAV:DATA?", datatype="B", container=np.array)
        raw_all.append(chunk)

        print(f"  Blok {start}–{end} ontvangen ({len(chunk)} punten)", end="\r")
        start += chunk_size

    print()

    raw        = np.concatenate(raw_all)
    voltage    = (raw.astype(float) - yorig - yref) * yinc
    time_array = np.arange(len(voltage)) * xinc + xorig

    print(f"Totaal ontvangen datapunten    : {len(voltage)}")
    print(f"Totale meetduur                : {time_array[-1] - time_array[0]:.6e} s")
    print(f"Tijdstap (XINC)                : {xinc:.6e} s")

    return time_array, voltage


def find_peaks_in_data(time, voltage,
                       smooth_window=21,
                       polyorder=2,
                       min_prominence=0.002,
                       min_distance_s=0.05):
    """
    Detecteert alleen fysisch betekenisvolle pieken.

    De originele RAW-data wordt NIET gewijzigd.
    Een gefilterde kopie wordt uitsluitend gebruikt voor piekdetectie.

    Parameters
    ----------
    smooth_window   : int   – breedte Savitzky-Golay filter (oneven, groter = meer ruisonderdrukking)
    polyorder       : int   – polynoomorde voor het filter
    min_prominence  : float – minimale hoogte van een piek boven zijn omgeving (Volt)
    min_distance_s  : float – minimale tijd tussen twee opeenvolgende pieken (seconden)
    """
    dt = np.median(np.diff(time))
    min_distance_pts = max(1, int(min_distance_s / dt))

    if smooth_window >= len(voltage):
        smooth_window = len(voltage) - 1
    if smooth_window % 2 == 0:
        smooth_window -= 1
    if smooth_window <= polyorder:
        voltage_filtered = voltage.copy()
    else:
        voltage_filtered = savgol_filter(voltage,
                                         window_length=smooth_window,
                                         polyorder=polyorder)

    peaks, properties = find_peaks(voltage_filtered,
                                   prominence=min_prominence,
                                   distance=min_distance_pts)

    return peaks, properties, voltage_filtered


def save_measurement_raw(time, voltage, params, peaks, measurement_number,
                         base_folder="measurements"):
    import os

    if not os.path.exists(base_folder):
        os.makedirs(base_folder)

    timestamp       = datetime.now().strftime("%Y%m%d_%H%M%S")
    data_filename   = f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_RAW_data.csv"
    params_filename = f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_params.csv"
    peaks_filename  = f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_peaks.csv"

    df_data = pd.DataFrame({"Time": time, "Voltage": voltage})
    df_data.to_csv(data_filename, index=False)
    print(f"\nData opgeslagen      : {data_filename}  ({len(df_data)} punten)")

    df_params = pd.DataFrame([params])
    df_params.to_csv(params_filename, index=False)
    print(f"Parameters opgeslagen: {params_filename}")

    peaks_data = pd.DataFrame({
        'Peak_Index'  : peaks,
        'Peak_Time'   : time[peaks],
        'Peak_Voltage': voltage[peaks]
    })
    peaks_data.to_csv(peaks_filename, index=False)
    print(f"Pieken opgeslagen    : {peaks_filename}  ({len(peaks)} pieken)")

    return data_filename, params_filename, peaks_filename


def plot_measurement_raw(time, voltage, voltage_filtered, peaks, params,
                         measurement_number):
    """
    Plot de ongewijzigde RAW-data met x-as exact gelijk aan het oscilloscoop-scherm.
    """
    time_range = time[-1] - time[0]
    if time_range < 1e-3:
        factor, time_unit = 1e6, "μs"
    elif time_range < 1:
        factor, time_unit = 1e3, "ms"
    else:
        factor, time_unit = 1, "s"

    time_display = time * factor

    timebase = params['timebase']
    x_center = params['trigger_offset']
    x_start  = (x_center - 5 * timebase) * factor
    x_end    = (x_center + 5 * timebase) * factor

    fig, ax = plt.subplots(figsize=(14, 6))

    ax.plot(time_display, voltage,
            color="lightsteelblue", linewidth=0.5,
            label="CH1 RAW-data")
    ax.plot(time_display, voltage_filtered,
            color="blue", linewidth=1.2,
            label="Gefilterd signaal (piekdetectie)")
    ax.plot(time_display[peaks], voltage[peaks],
            "rx", markersize=8, markeredgewidth=1.5,
            label=f"Pieken ({len(peaks)})")

    ax.set_xlim(x_start, x_end)
    ax.set_xticks(np.linspace(x_start, x_end, 11))

    ax.set_xlabel(f"Tijd ({time_unit})")
    ax.set_ylabel("Spanning (V)")
    ax.set_title(
        f"Meting {measurement_number} — RAW-data\n"
        f"Timebase: {timebase} s/div | "
        f"Verticale schaal: {params['vertical_scale']}\n"
        f"Datapunten: {len(voltage)} | Getelde pieken: {len(peaks)}"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    return fig


if __name__ == "__main__":
    scope = connect()

    print("\n=== RAW DATA METING (GEEN COMPRESSIE) ===\n")

    t, v   = read_channel_raw(scope, "CHAN1", chunk_size=10000)
    params = get_scope_parameters(scope, "CHAN1")

    peaks, properties, v_filtered = find_peaks_in_data(
        t, v,
        smooth_window=21,
        polyorder=2,
        min_prominence=0.002,
        min_distance_s=0.05
    )

    data_file, params_file, peaks_file = save_measurement_raw(
        t, v, params, peaks, 1, "measurements"
    )

    scope.close()

    plot_measurement_raw(t, v, v_filtered, peaks, params, 1)

    print(f"\n✓ Meting voltooid! Pieken: {peaks_file}")


In [ ]:
import pyvisa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.signal import find_peaks, savgol_filter


def connect():
    rm = pyvisa.ResourceManager()
    print("Gevonden VISA-resources:", rm.list_resources())

    resources = [r for r in rm.list_resources() if "USB" in r]
    if not resources:
        raise RuntimeError("Geen USB-apparaten gevonden")

    scope = rm.open_resource(resources[0])
    scope.timeout = 30000  # ms per blok
    print("Verbonden met:", scope.query("*IDN?").strip())
    return scope


def get_scope_parameters(scope, channel="CHAN1"):
    params = {
        'timestamp':         datetime.now().isoformat(),
        'channel':           channel,
        'xinc':              float(scope.query(":WAV:XINC?")),
        'xorig':             float(scope.query(":WAV:XOR?")),
        'yinc':              float(scope.query(":WAV:YINC?")),
        'yorig':             float(scope.query(":WAV:YOR?")),
        'yref':              float(scope.query(":WAV:YREF?")),
        'timebase':          float(scope.query(":TIM:SCAL?")),
        'trigger_offset':    float(scope.query(":TIM:OFFS?")),
        'vertical_scale':    scope.query(f":{channel}:SCAL?").strip(),
        'vertical_offset':   scope.query(f":{channel}:OFFS?").strip(),
        'coupling':          scope.query(f":{channel}:COUP?").strip(),
        'probe_attenuation': scope.query(f":{channel}:PROB?").strip(),
        'memory_depth':      scope.query(":ACQ:MDEP?").strip(),
    }
    return params


def read_channel_raw(scope, channel="CHAN1", chunk_size=10000):
    """
    Lees volledige RAW memory depth van Rigol DS2102A in blokken.
    chunk_size : aantal punten per blok (verlaag naar 5000 bij timeout)
    """
    scope.write(f":WAV:SOUR {channel}")
    scope.write(":WAV:MODE RAW")
    scope.write(":WAV:FORM BYTE")

    # Scaling-parameters
    xinc  = float(scope.query(":WAV:XINC?"))
    xorig = float(scope.query(":WAV:XOR?"))
    yinc  = float(scope.query(":WAV:YINC?"))
    yorig = float(scope.query(":WAV:YOR?"))
    yref  = float(scope.query(":WAV:YREF?"))

    # Rigol DS2102A: memory depth via :ACQ:MDEP
    mdep_raw = scope.query(":ACQ:MDEP?").strip()
    print(f"ACQ:MDEP antwoord: '{mdep_raw}'")

    if mdep_raw.upper() == "AUTO":
        # AUTO: scope kiest zelf — lees hoeveel punten er werkelijk zijn
        # via een eerste volledige opvraag zonder STAR/STOP beperking
        scope.write(":WAV:STAR 1")
        scope.write(":WAV:STOP 1400")
        probe = scope.query_binary_values(":WAV:DATA?", datatype="B", container=np.array)
        total_points = len(probe)
        print(f"Memory depth: AUTO → werkelijk {total_points} punten")
    else:
        total_points = int(float(mdep_raw))
        print(f"Memory depth: {total_points} punten")

    # Haal data op in blokken
    raw_all = []
    start   = 1

    while start <= total_points:
        end = min(start + chunk_size - 1, total_points)
        scope.write(f":WAV:STAR {start}")
        scope.write(f":WAV:STOP {end}")

        chunk = scope.query_binary_values(":WAV:DATA?", datatype="B", container=np.array)
        raw_all.append(chunk)

        print(f"  Blok {start}–{end} ontvangen ({len(chunk)} punten)", end="\r")
        start += chunk_size

    print()

    raw        = np.concatenate(raw_all)
    voltage    = (raw.astype(float) - yorig - yref) * yinc
    time_array = np.arange(len(voltage)) * xinc + xorig

    print(f"Totaal ontvangen datapunten    : {len(voltage)}")
    print(f"Totale meetduur                : {time_array[-1] - time_array[0]:.6e} s")
    print(f"Tijdstap (XINC)                : {xinc:.6e} s")

    return time_array, voltage


def find_peaks_in_data(time, voltage,
                       smooth_window=21,
                       polyorder=2,
                       min_prominence=0.002,
                       min_distance_s=0.05):
    """
    Detecteert alleen fysisch betekenisvolle pieken.

    De originele RAW-data wordt NIET gewijzigd.
    Een gefilterde kopie wordt uitsluitend gebruikt voor piekdetectie.

    Parameters
    ----------
    smooth_window   : int   – breedte Savitzky-Golay filter (oneven, groter = meer ruisonderdrukking)
    polyorder       : int   – polynoomorde voor het filter
    min_prominence  : float – minimale hoogte van een piek boven zijn omgeving (Volt)
    min_distance_s  : float – minimale tijd tussen twee opeenvolgende pieken (seconden)
    """
    dt = np.median(np.diff(time))
    min_distance_pts = max(1, int(min_distance_s / dt))

    if smooth_window >= len(voltage):
        smooth_window = len(voltage) - 1
    if smooth_window % 2 == 0:
        smooth_window -= 1
    if smooth_window <= polyorder:
        voltage_filtered = voltage.copy()
    else:
        voltage_filtered = savgol_filter(voltage,
                                         window_length=smooth_window,
                                         polyorder=polyorder)

    peaks, properties = find_peaks(voltage_filtered,
                                   prominence=min_prominence,
                                   distance=min_distance_pts)

    return peaks, properties, voltage_filtered


def save_measurement_raw(time, voltage, params, peaks, measurement_number,
                         base_folder="measurements"):
    import os

    if not os.path.exists(base_folder):
        os.makedirs(base_folder)

    timestamp       = datetime.now().strftime("%Y%m%d_%H%M%S")
    data_filename   = f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_RAW_data.csv"
    params_filename = f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_params.csv"
    peaks_filename  = f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_peaks.csv"

    df_data = pd.DataFrame({"Time": time, "Voltage": voltage})
    df_data.to_csv(data_filename, index=False)
    print(f"\nData opgeslagen      : {data_filename}  ({len(df_data)} punten)")

    df_params = pd.DataFrame([params])
    df_params.to_csv(params_filename, index=False)
    print(f"Parameters opgeslagen: {params_filename}")

    peaks_data = pd.DataFrame({
        'Peak_Index'  : peaks,
        'Peak_Time'   : time[peaks],
        'Peak_Voltage': voltage[peaks]
    })
    peaks_data.to_csv(peaks_filename, index=False)
    print(f"Pieken opgeslagen    : {peaks_filename}  ({len(peaks)} pieken)")

    return data_filename, params_filename, peaks_filename


def plot_measurement_raw(time, voltage, voltage_filtered, peaks, params,
                         measurement_number):
    """
    Plot de ongewijzigde RAW-data met x-as exact gelijk aan het oscilloscoop-scherm.
    """
    time_range = time[-1] - time[0]
    if time_range < 1e-3:
        factor, time_unit = 1e6, "μs"
    elif time_range < 1:
        factor, time_unit = 1e3, "ms"
    else:
        factor, time_unit = 1, "s"

    time_display = time * factor

    timebase = params['timebase']
    x_center = params['trigger_offset']
    x_start  = (x_center - 5 * timebase) * factor
    x_end    = (x_center + 5 * timebase) * factor

    fig, ax = plt.subplots(figsize=(14, 6))

    ax.plot(time_display, voltage,
            color="lightsteelblue", linewidth=0.5,
            label="CH1 RAW-data")
    ax.plot(time_display, voltage_filtered,
            color="blue", linewidth=1.2,
            label="Gefilterd signaal (piekdetectie)")
    ax.plot(time_display[peaks], voltage[peaks],
            "rx", markersize=8, markeredgewidth=1.5,
            label=f"Pieken ({len(peaks)})")

    ax.set_xlim(x_start, x_end)
    ax.set_xticks(np.linspace(x_start, x_end, 11))

    ax.set_xlabel(f"Tijd ({time_unit})")
    ax.set_ylabel("Spanning (V)")
    ax.set_title(
        f"Meting {measurement_number} — RAW-data\n"
        f"Timebase: {timebase} s/div | "
        f"Verticale schaal: {params['vertical_scale']}\n"
        f"Datapunten: {len(voltage)} | Getelde pieken: {len(peaks)}"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    return fig


if __name__ == "__main__":
    scope = connect()

    print("\n=== RAW DATA METING (GEEN COMPRESSIE) ===\n")

    t, v   = read_channel_raw(scope, "CHAN1", chunk_size=10000)
    params = get_scope_parameters(scope, "CHAN1")

    peaks, properties, v_filtered = find_peaks_in_data(
        t, v,
        smooth_window=21,
        polyorder=2,
        min_prominence=0.002,
        min_distance_s=0.05
    )

    data_file, params_file, peaks_file = save_measurement_raw(
        t, v, params, peaks, 1, "measurements"
    )

    scope.close()

    plot_measurement_raw(t, v, v_filtered, peaks, params, 1)

    print(f"\n✓ Meting voltooid! Pieken: {peaks_file}")


In [ ]:
import pyvisa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.signal import find_peaks, savgol_filter


def connect():
    rm = pyvisa.ResourceManager()
    print("Gevonden VISA-resources:", rm.list_resources())
    resources = [r for r in rm.list_resources() if "USB" in r]
    if not resources:
        raise RuntimeError("Geen USB-apparaten gevonden")
    scope = rm.open_resource(resources[0])
    scope.timeout = 30000
    print("Verbonden met:", scope.query("*IDN?").strip())
    return scope


def get_scope_parameters(scope, channel="CHAN1"):
    params = {
        'timestamp':         datetime.now().isoformat(),
        'channel':           channel,
        'xinc':              float(scope.query(":WAV:XINC?")),
        'xorig':             float(scope.query(":WAV:XOR?")),
        'yinc':              float(scope.query(":WAV:YINC?")),
        'yorig':             float(scope.query(":WAV:YOR?")),
        'yref':              float(scope.query(":WAV:YREF?")),
        'timebase':          float(scope.query(":TIM:SCAL?")),
        'trigger_offset':    float(scope.query(":TIM:OFFS?")),
        'vertical_scale':    scope.query(f":{channel}:SCAL?").strip(),
        'vertical_offset':   scope.query(f":{channel}:OFFS?").strip(),
        'coupling':          scope.query(f":{channel}:COUP?").strip(),
        'probe_attenuation': scope.query(f":{channel}:PROB?").strip(),
        'memory_depth':      scope.query(":ACQ:MDEP?").strip(),
    }
    return params


def read_channel_raw(scope, channel="CHAN1", chunk_size=10000):
    scope.write(f":WAV:SOUR {channel}")
    scope.write(":WAV:MODE RAW")
    scope.write(":WAV:FORM BYTE")

    xinc  = float(scope.query(":WAV:XINC?"))
    xorig = float(scope.query(":WAV:XOR?"))
    yinc  = float(scope.query(":WAV:YINC?"))
    yorig = float(scope.query(":WAV:YOR?"))
    yref  = float(scope.query(":WAV:YREF?"))

    mdep_raw = scope.query(":ACQ:MDEP?").strip()
    print(f"ACQ:MDEP antwoord: '{mdep_raw}'")

    if mdep_raw.upper() == "AUTO":
        scope.write(":WAV:STAR 1")
        scope.write(":WAV:STOP 1400")
        probe = scope.query_binary_values(":WAV:DATA?", datatype="B", container=np.array)
        total_points = len(probe)
        print(f"Memory depth: AUTO → werkelijk {total_points} punten")
    else:
        total_points = int(float(mdep_raw))
        print(f"Memory depth: {total_points} punten")

    raw_all = []
    start   = 1
    while start <= total_points:
        end = min(start + chunk_size - 1, total_points)
        scope.write(f":WAV:STAR {start}")
        scope.write(f":WAV:STOP {end}")
        chunk = scope.query_binary_values(":WAV:DATA?", datatype="B", container=np.array)
        raw_all.append(chunk)
        print(f"  Blok {start}–{end} ontvangen ({len(chunk)} punten)", end="\r")
        start += chunk_size

    print()
    raw        = np.concatenate(raw_all)
    voltage    = (raw.astype(float) - yorig - yref) * yinc
    time_array = np.arange(len(voltage)) * xinc + xorig

    print(f"Totaal ontvangen datapunten    : {len(voltage)}")
    print(f"Totale meetduur                : {time_array[-1] - time_array[0]:.6e} s")
    print(f"Tijdstap (XINC)                : {xinc:.6e} s")
    return time_array, voltage


def find_peaks_in_data(time, voltage,
                       smooth_window=21,
                       polyorder=2,
                       min_prominence=0.002,
                       min_distance_s=0.05):
    """
    Detecteert fysisch betekenisvolle pieken via Savitzky-Golay filtering.
    De originele RAW-data wordt NIET gewijzigd.

    Parameters
    ----------
    smooth_window   : int   – breedte Savitzky-Golay filter (oneven)
    polyorder       : int   – polynoomorde voor het filter
    min_prominence  : float – minimale hoogte boven omgeving (Volt)
    min_distance_s  : float – minimale tijd tussen twee pieken (seconden)
    """
    dt = np.median(np.diff(time))
    min_distance_pts = max(1, int(min_distance_s / dt))

    if smooth_window >= len(voltage):
        smooth_window = len(voltage) - 1
    if smooth_window % 2 == 0:
        smooth_window -= 1
    if smooth_window <= polyorder:
        voltage_filtered = voltage.copy()
    else:
        voltage_filtered = savgol_filter(voltage,
                                         window_length=smooth_window,
                                         polyorder=polyorder)

    peaks, properties = find_peaks(voltage_filtered,
                                   prominence=min_prominence,
                                   distance=min_distance_pts)
    return peaks, properties, voltage_filtered


def save_measurement(time, voltage, params, peaks, measurement_number,
                     base_folder="measurements"):
    import os
    if not os.path.exists(base_folder):
        os.makedirs(base_folder)

    timestamp      = datetime.now().strftime("%Y%m%d_%H%M%S")
    data_filename  = f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_RAW_data.csv"
    params_filename= f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_params.csv"
    peaks_filename = f"{base_folder}/measurement_{measurement_number:03d}_{timestamp}_peaks.csv"

    pd.DataFrame({"Time": time, "Voltage": voltage}).to_csv(data_filename, index=False)
    print(f"\nData opgeslagen      : {data_filename}  ({len(time)} punten)")

    pd.DataFrame([params]).to_csv(params_filename, index=False)
    print(f"Parameters           : {params_filename}")

    pd.DataFrame({
        'Peak_Index'  : peaks,
        'Peak_Time'   : time[peaks],
        'Peak_Voltage': voltage[peaks]
    }).to_csv(peaks_filename, index=False)
    print(f"Pieken               : {peaks_filename}  ({len(peaks)} pieken)")

    return data_filename, params_filename, peaks_filename


def plot_measurement(time, voltage, voltage_filtered, peaks, params,
                     measurement_number):
    time_range = time[-1] - time[0]
    if time_range < 1e-3:
        factor, time_unit = 1e6, "μs"
    elif time_range < 1:
        factor, time_unit = 1e3, "ms"
    else:
        factor, time_unit = 1, "s"

    timebase = params['timebase']
    x_center = params['trigger_offset']
    x_start = -1.33   # in seconden
    x_end   =  -1.16

    fig, ax = plt.subplots(figsize=(14, 6))

    ax.plot(time * factor, voltage,
            color="lightsteelblue", linewidth=0.5, label="RAW-data")
    ax.plot(time * factor, voltage_filtered,
            color="blue", linewidth=1.2, label="Gefilterd signaal")
    ax.plot(time[peaks] * factor, voltage[peaks],
            "rx", markersize=8, markeredgewidth=1.5,
            label=f"Pieken ({len(peaks)})")

    ax.set_xlim(x_start, x_end)
    ax.set_xticks(np.linspace(x_start, x_end, 11))
    ax.set_xlabel(f"Tijd ({time_unit})")
    ax.set_ylabel("Spanning (V)")
    ax.set_title(
        f"Meting {measurement_number} — Timebase: {timebase} s/div | "
        f"Verticale schaal: {params['vertical_scale']}\n"
        f"Datapunten: {len(voltage)} | Pieken: {len(peaks)}"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    return fig


if __name__ == "__main__":
    scope = connect()

    print("\n=== RAW DATA METING (GEEN COMPRESSIE) ===\n")

    t, v   = read_channel_raw(scope, "CHAN1", chunk_size=10000)
    params = get_scope_parameters(scope, "CHAN1")
    scope.close()

    peaks, properties, v_filtered = find_peaks_in_data(
        t, v,
        smooth_window=5,        # was 21 → veel smaller zodat pieken zichtbaar blijven
        polyorder=2,
        min_prominence=0.0005,   # was 0.002 → lager zodat kleine pieken ook tellen
        min_distance_s=0.005   )

    save_measurement(t, v, params, peaks, 1, "measurements")

    plot_measurement(t, v, v_filtered, peaks, params, 1)

    print(f"\n✓ Meting voltooid! {len(peaks)} pieken gevonden.")
    
